## Resent50 training with PyTorch

In [ ]:
# @title
!pip install openimages

In [ ]:
data_folder = "data"
sample_count = 1000
# find all classes and mappings here: https://deeplearning.cms.waikato.ac.nz/user-guide/class-maps/IMAGENET/
class_to_index = {"Orange": 950, "Pillow": 721, "Vase": 883}

In [ ]:
from openimages.download import download_dataset
import os

def download_openimages(data_folder, sample_count, classes):
  if not os.path.exists(data_folder):
      os.makedirs(data_folder)
  # Download equal amount of images per class if possible
  sample_count_per_class = sample_count // len(classes)
  download_dataset(data_folder, classes, limit=sample_count_per_class)

download_openimages(data_folder, sample_count, list(class_to_index.keys()))

KeyboardInterrupt: 

In [ ]:
import torch
from torchvision import transforms, models, datasets

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = models.resnet50(pretrained = True).to(device)
model.eval()
all_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
])

In [ ]:
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

class ImageDataset(Dataset):
  def __init__(self, data_folder, class_to_index, transform):
    self.transform = transform
    self.paths_with_class = self.get_paths_with_class(data_folder, class_to_index)

  def __len__(self):
    return len(self.paths_with_class)

  def __getitem__(self, index):
    path, image_class_index = self.paths_with_class[index]
    compatible_image = self.get_compatible_image(path)
    return (compatible_image, image_class_index)

  def get_paths_with_class(self, data_folder, class_to_index):
    paths_with_class_index = []
    for class_name in class_to_index.keys():
      image_paths = glob.glob("{}/{}/images/*.jpg".format(data_folder, class_name.lower()))
      # make (path_to_image, true_class_index) pairs for shuffling and unequal image count per class
      paths_with_class_index += map(lambda paths: (paths, class_to_index[class_name]), list(image_paths))
    return paths_with_class_index

  def get_compatible_image(self, full_path):
    # have only 3 channels per pixel
    image = Image.open(full_path).convert('RGB')
    transformed_image = self.transform(image)
    return transformed_image

dataset = ImageDataset(data_folder, class_to_index, all_transforms)

In [ ]:
import numpy as np

def get_top_with_labels(image_folder, dataset, top_count=3, batch_size=50):
  dataset_size = len(dataset)
  top_probs = np.empty(dataset_size)
  guess_class_index = np.empty(dataset_size, dtype=np.int16)
  true_class_index = np.empty(dataset_size, dtype=np.int16)
  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4)
  i = 0
  # per iteration put a batch_size of top probabilities with guessed class and base true class indices
  for input, true_index in dataloader:
    input = input.to(device)
    outputs = model(input)
    probs = torch.sigmoid(outputs)
    next_i = i + batch_size
    guess_class_index[i : next_i] = torch.argmax(probs, dim=1).cpu()
    # without detach it causes RuntimeError and the error suggests using it
    top_probs[i : next_i] = probs.detach().cpu().numpy()[:, guess_class_index[i]]
    true_class_index[i : next_i] = true_index
    i = next_i
  return (top_probs, guess_class_index, true_class_index)

results = get_top_with_labels(data_folder, dataset)
results

(array([0.99999785, 0.99988902, 0.99744499, 0.99998426, 1.        ,
        0.96827191, 0.99977118, 0.99998605, 0.99999905, 0.99999988,
        0.99999619, 1.        , 0.99999964, 1.        , 1.        ,
        0.9999994 , 0.99997604, 0.99997032, 1.        , 1.        ,
        0.99999976, 1.        , 0.97741151, 0.99999952, 0.99964786,
        0.99999988, 0.99978286, 0.99710542, 0.99998748, 0.99970943,
        1.        , 1.        , 0.99910152, 0.9999907 , 0.99989355,
        0.99931824, 0.99999738, 0.99987125, 0.99998367, 0.99999893,
        0.99994063, 0.99950886, 0.99973923, 0.99973148, 0.99999022,
        1.        , 0.99954456, 1.        , 1.        , 0.99999416,
        0.99999964, 0.99999571, 0.99995124, 1.        , 0.99436861,
        0.53351659, 0.99999261, 0.99899286, 0.99846315, 1.        ,
        1.        , 0.99983478, 0.99999022, 0.99769419, 0.99353856,
        0.9999913 , 1.        , 0.99999988, 0.99998736, 0.99999595,
        0.99994648, 0.9999969 , 0.99999094, 0.99

In [ ]:
import math

def get_metrics(results, for_class_index, threshold=0.5):
  top_probs, guess_class_index, true_class_index = results
  # Cached masks
  confident_guess = top_probs > threshold
  guess_for_class_index = guess_class_index == for_class_index
  true_for_class_index = true_class_index == for_class_index
  positive_guess = confident_guess & guess_for_class_index
  tp = (
      positive_guess
      & true_for_class_index
      ).sum()
  fp = (
      positive_guess
      & ~true_for_class_index
      ).sum()
  fn = (
      ~positive_guess
      & true_for_class_index
      ).sum()
  size = len(top_probs)
  print(f"size={size}\ntp={tp}\nfp={fp}\nfn={fn}\ntn={size - tp - fp - fn}")
  accuracy = (size - fp - fn) / size
  precision = tp / (tp + fp)
  recall = tp / (tp + fn)
  f2 = (2 * precision * recall) / (precision + recall)
  return (accuracy, precision, recall, f2)

def print_metrics(results, for_class, class_to_index, threshold=0.5):
  class_index = class_to_index[for_class]
  accuracy, precision, recall, f2 = get_metrics(results, class_index, threshold)
  print(f"{for_class}, index = {class_index}, threshold = {threshold}:\n\
        accuracy: {accuracy:.2f}\n\
        precision: {precision:.2f}\n\
        recall: {recall:.2f}\n\
        f2: {f2:.2f}"
  )

for_classes = ["Vase"] * 3
thresholds = (0.2, 0.5, 0.8)
for for_class, threshold in zip(for_classes, thresholds):
  print_metrics(results, for_class, class_to_index, threshold)

size=999
tp=148
fp=1
fn=185
tn=665
Vase, index = 883, threshold = 0.2:
        accuracy: 0.81
        precision: 0.99
        recall: 0.44
        f2: 0.61
size=999
tp=137
fp=1
fn=196
tn=665
Vase, index = 883, threshold = 0.5:
        accuracy: 0.80
        precision: 0.99
        recall: 0.41
        f2: 0.58
size=999
tp=112
fp=1
fn=221
tn=665
Vase, index = 883, threshold = 0.8:
        accuracy: 0.78
        precision: 0.99
        recall: 0.34
        f2: 0.50
